In [ ]:
from lxml import etree
from rapidfuzz import fuzz
from pathlib import Path
import json

# =====================================================
# UTILS
# =====================================================

def normalize(text):
    return text.lower().strip() if text else ""


# =====================================================
# 1. EXTRACTION GLOBALE DES BALISES
# =====================================================

def extract_global_tags(xml_string: str):
    root = etree.fromstring(xml_string.encode())

    ns = {"tei": root.nsmap.get(None)} if None in root.nsmap else {}

    def xp(path):
        return root.xpath(path, namespaces=ns)
#Éléments à évaluer au niveau 2 (à commenter ou décommenter)
    definition = xp(".//tei:def/text()" if ns else ".//def/text()")
    traduction = xp(".//tei:cit[@type='translation']/tei:quote/text()" if ns else ".//cit[@type='translation']/quote/text()")
    exemple = xp(".//tei:cit[@type='example']/tei:quote/text()" if ns else ".//cit[@type='example']/quote/text()")
    renvoi = xp(".//tei:xr/text()" if ns else ".//xr/text()")
    reference = xp(".//tei:ref/text()" if ns else ".//ref/text()")
    bibliographie = xp(".//tei:bibl/text()" if ns else ".//bibl/text()")
    auteur = xp(".//tei:author/text()" if ns else ".//author/text()")

#Éléments à évaluer au niveau 3 (à commenter ou décommenter)
    #encycl = xp(".//tei:seg/text()" if ns else ".//seg/text()")
    #etym = xp(".//tei:etym/text()" if ns else ".//etym/text()")
    #glose = xp(".//tei:gloss/text()" if ns else ".//gloss/text()")
    #oRef = xp(".//tei:oRef/text()" if ns else ".//oRef/text()")
    #foreign = xp(".//tei:foreign/text()" if ns else ".//foreign/text()")
    #lang = xp(".//tei:lang/text()" if ns else ".//lang/text()")
    
    return {
#Compter le nombre d'occurrences des éléments du niveau 2 (à commenter ou décommenter)
        "definition": len(definition),
        "traduction": len(traduction),
        "exemple": len(exemple),
        "renvoi": len(renvoi),
        "reference": len(reference),
        "bibliographie": len(bibliographie),
        "auteur": len(auteur),
#Compter le nombre d'occurrences des éléments du niveau 3 (à commenter ou décommenter)
        #"encycl": len(encycl),
        #"etym": len(etym),
        #"glose": len(glose),
        #"oRef": len(oRef),
        #"foreign": len(foreign),
        #"lang": len(lang),
    }


# =====================================================
# 2. SCORING ENTRIES
# =====================================================

def compute_scores(alignment):
    tp = len(alignment["matches"])
    fp = len(alignment["insertions"]) + len(alignment["replacements"])
    fn = len(alignment["deletions"]) + len(alignment["replacements"])

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": 2 * precision * recall / (precision + recall) if (precision + recall) else 0,
        "pred_count": tp + fp,
        "gold_count": tp + fn,
    }


# =====================================================
# 3. SCORING BALISES GLOBALES
# =====================================================

def compute_tag_scores(pred_tags, gold_tags):
    scores = {}

    for tag in pred_tags.keys():
        pred = pred_tags[tag]
        gold = gold_tags[tag]

        tp = min(pred, gold)
        fp = max(0, pred - gold)
        fn = max(0, gold - pred)

        precision = tp / (tp + fp) if (tp + fp) else 0
        recall = tp / (tp + fn) if (tp + fn) else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

        scores[tag] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "pred_count": pred,
            "gold_count": gold
        }

    return scores


# =====================================================
# 4. PIPELINE GLOBAL
# =====================================================

def evaluation(pred_path, gold_path, output_path):
    pred_xml = Path(pred_path).read_text(encoding="utf-8")
    gold_xml = Path(gold_path).read_text(encoding="utf-8")

    # --- TAGS ---
    pred_tags = extract_global_tags(pred_xml)
    gold_tags = extract_global_tags(gold_xml)

    # --- FINAL REPORT ---
    final_report = {
        "tags": compute_tag_scores(pred_tags, gold_tags)
    }

    # --- SAVE ---
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_report, f, indent=2, ensure_ascii=False)

    return final_report

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

# =====================================================
# TRAITEMENT GENERIQUE D'UN "NIVEAU" (niveau 2, niveau 2.2, etc.)
# =====================================================

# Pour chaque document x prompt x modele, on produit un fichier JSON d'evaluation dans eval_dir, sans jamais arrêter le traitement en cas de fichier manquant ou mal formé.
# Dans les paramètres de la fonction on décide quel est le niveau de la vérité de terrain (chemin relatif) à partir de laquelle on va obtenir les métriques souhaitées.
def traiter_niveau(niveau_label, prompts_dir, output_dir, eval_dir, gt_dir="gt/niveau2", inputs_dir="input"):
    Path(eval_dir).mkdir(parents=True, exist_ok=True)

    prompts_n = sorted(Path(prompts_dir).glob("*.txt"))
    models_n = sorted([p for p in Path(output_dir).iterdir() if p.is_dir()])
    inputs_n = sorted(Path(inputs_dir).glob("*.txt"))

    print(f"\n########## NIVEAU: {niveau_label} ##########")
    print("prompts trouves:", [p.name for p in prompts_n])
    print("models trouves:", [m.name for m in models_n])
    print("inputs trouves:", [i.name for i in inputs_n])

    fichiers_mal_formes_n = []

    for input_file in inputs_n:
        doc_name = input_file.stem
        gt_file = Path(gt_dir) / f"{doc_name}.xml"
        print(f"\n=== [{niveau_label}] DOCUMENT: {doc_name} ===")

        for prompt in prompts_n:
            print(f"  -> PROMPT: {prompt.stem}")

            for model_dir in models_n:
                output_file = model_dir / prompt.stem / f"{doc_name}.xml"

                if not output_file.exists():
                    print(f"     -> {model_dir.name}: fichier manquant ({output_file}), on continue")
                    continue

                print(f"     -> {model_dir.name}")

                # Verification prealable : XML bien forme ?
                try:
                    ET.parse(output_file)
                except ET.ParseError as e:
                    message = f"     FICHIER MAL FORME (XML invalide) : {output_file} - {e}"
                    print(message)
                    fichiers_mal_formes_n.append(str(output_file))
                    continue

                # Evaluation, sécurisée
                try:
                    evaluation(
                        output_file,
                        gt_file,
                        output_path=str(Path(eval_dir) / f"{model_dir.name}_{prompt.stem}_{doc_name}.json")
                    )
                except Exception as e:
                    message = f"     ERREUR lors de l'evaluation de {output_file} : {e}"
                    print(message)
                    fichiers_mal_formes_n.append(str(output_file))
                    continue

    return {
        "niveau": niveau_label,
        "prompts": prompts_n,
        "models": models_n,
        "inputs": inputs_n,
        "eval_dir": eval_dir,
        "fichiers_mal_formes": fichiers_mal_formes_n,
    }


# --- Lancement de la fonction traiter_niveau() pour les deux niveaux (prompt unique et 2ème prompt du prompt chaining) qu'on va comparer plus tard dans un même tableau ---
#Configuration du premier niveau à comparer
config_niveau1 = traiter_niveau(
    niveau_label="niveau2.3",
    prompts_dir="prompts/niveau2.3",
    output_dir="output/niveau2.3",
    eval_dir="evaluation/niveau2.3",
)

#Configuration du deuxième niveau à comparer
config_niveau2 = traiter_niveau(
    niveau_label="niveau2.5",
    prompts_dir="prompts/niveau2.5",
    output_dir="output/niveau2.5",
    eval_dir="evaluation/niveau2.5",
)

configs_niveaux = [config_niveau1, config_niveau2]

# --- Recapitulatif final ---
total_mal_formes = sum(len(c["fichiers_mal_formes"]) for c in configs_niveaux)
if total_mal_formes:
    print(f"\n\n{total_mal_formes} fichier(s) mal forme(s) ou en erreur au total, ignores pendant le traitement :")
    for c in configs_niveaux:
        for f in c["fichiers_mal_formes"]:
            print(f"   - [{c['niveau']}] {f}")
else:
    print("\n\nAucun fichier mal forme detecte.")


########## NIVEAU: niveau2.6 ##########
prompts trouves: ['FS-R.txt', 'FS.txt', 'ZS.txt']
models trouves: ['gemini-3.1-flash-lite-preview', 'gemma-3-27b-it', 'gpt-5-mini', 'gpt-5.4-mini']
inputs trouves: ['TR1_p2001-2002.txt', 'TR1_p453-454.txt', 'TR2_p1785-1786.txt', 'TR2_p37-38.txt', 'TR3_p5-6.txt', 'TR3_p7-8.txt', 'TR4_p131-132.txt', 'TR5_p489-490.txt', 'TR5_p505-506.txt', 'TR6_p1003-1004.txt']

=== [niveau2.6] DOCUMENT: TR1_p2001-2002 ===
  -> PROMPT: FS-R
     -> gemini-3.1-flash-lite-preview
     -> gemma-3-27b-it
     -> gpt-5-mini
     -> gpt-5.4-mini
  -> PROMPT: FS
     -> gemini-3.1-flash-lite-preview
     -> gemma-3-27b-it
     -> gpt-5-mini
     -> gpt-5.4-mini
  -> PROMPT: ZS
     -> gemini-3.1-flash-lite-preview
     -> gemma-3-27b-it
     -> gpt-5-mini
     -> gpt-5.4-mini

=== [niveau2.6] DOCUMENT: TR1_p453-454 ===
  -> PROMPT: FS-R
     -> gemini-3.1-flash-lite-preview
     -> gemma-3-27b-it
     -> gpt-5-mini
     -> gpt-5.4-mini
  -> PROMPT: FS
     -> gemini-3.1-f

In [ ]:
from collections import defaultdict
import json
import pandas as pd

# =====================================================
# MICRO F-MESURE PAR MODELE, TYPE DE PROMPT ET NIVEAU
# =====================================================


def get_prompt_type(prompt_stem: str) -> str:
    return prompt_stem.split("_")[0]


def agreger_micro_f1(config):
    # Parcourir les JSON déjà produits pour un niveau donné et retourner les lignes agregées (une ligne par modele x type de prompt x tag).
    niveau_label = config["niveau"]
    eval_dir = Path(config["eval_dir"])
    prompts_n = config["prompts"]
    models_n = config["models"]
    inputs_n = config["inputs"]

    agg = defaultdict(lambda: defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0}))
    fichiers_ignores = []

    for input_file in inputs_n:
        doc_name = input_file.stem
        for prompt in prompts_n:
            prompt_type = get_prompt_type(prompt.stem)
            for model_dir in models_n:
                eval_file = eval_dir / f"{model_dir.name}_{prompt.stem}_{doc_name}.json"
                if not eval_file.exists():
                    continue

                try:
                    with open(eval_file, "r", encoding="utf-8") as f:
                        report = json.load(f)
                except (json.JSONDecodeError, OSError) as e:
                    print(f" FICHIER D'EVALUATION ILLISIBLE, ignore : {eval_file} - {e}")
                    fichiers_ignores.append(str(eval_file))
                    continue

                key = (model_dir.name, prompt_type)
                for tag, scores in report.get("tags", {}).items():
                    pred = scores["pred_count"]
                    gold = scores["gold_count"]
                    # tp=min(pred,gold) ; fp=max(0,pred-gold) ; fn=max(0,gold-pred)
                    # (coherent avec compute_tag_scores)
                    tp = min(pred, gold)
                    fp = max(0, pred - gold)
                    fn = max(0, gold - pred)
                    agg[key][tag]["tp"] += tp
                    agg[key][tag]["fp"] += fp
                    agg[key][tag]["fn"] += fn

    rows = []
    for (model, prompt_type), tags in agg.items():
        for tag, counts in tags.items():
            tp, fp, fn = counts["tp"], counts["fp"], counts["fn"]
            precision = tp / (tp + fp) if (tp + fp) else 0.0
            recall = tp / (tp + fn) if (tp + fn) else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
            rows.append({
                "modele": model,
                "type_prompt": prompt_type,
                "niveau": niveau_label,
                "tag": tag,
                "tp": tp, "fp": fp, "fn": fn,
                "precision_micro": round(precision, 4),
                "recall_micro": round(recall, 4),
                "f1_micro": round(f1, 4),
            })

    if fichiers_ignores:
        print(f"\n  [{niveau_label}] {len(fichiers_ignores)} fichier(s) d'evaluation ignore(s) (JSON illisible).")

    return rows


# --- Agregation pour chaque niveau, puis fusion ---
toutes_les_lignes = []
for config in configs_niveaux:
    toutes_les_lignes.extend(agreger_micro_f1(config))

#Dataframe avec toutes ces informations
df_micro = pd.DataFrame(toutes_les_lignes).sort_values(
    ["modele", "type_prompt", "niveau", "tag"]
).reset_index(drop=True)

for config in configs_niveaux:
    eval_dir = Path(config["eval_dir"])
    output_csv =  eval_dir/ "micro_f1_par_modele_prompt_niveau.csv"
    df_micro.to_csv(output_csv, index=False)
    print(f"Resultats exportes vers : {output_csv}")

df_micro

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Resultats exportes vers : evaluation/niveau2.6/micro_f1_par_modele_prompt_niveau.csv
Resultats exportes vers : evaluation/niveau2.8/micro_f1_par_modele_prompt_niveau.csv


,modele,type_prompt,niveau,tag,tp,fp,fn,precision_micro,recall_micro,f1_micro
0,gemini-3.1-flash-lite-preview,FS,niveau2.6,auteur,123,4,42,0.9685,0.7455,0.8425
1,gemini-3.1-flash-lite-preview,FS,niveau2.6,bibliographie,82,97,36,0.4581,0.6949,0.5522
2,gemini-3.1-flash-lite-preview,FS,niveau2.6,definition,182,151,1,0.5465,0.9945,0.7054
3,gemini-3.1-flash-lite-preview,FS,niveau2.6,exemple,179,46,62,0.7956,0.7427,0.7682
4,gemini-3.1-flash-lite-preview,FS,niveau2.6,reference,23,75,6,0.2347,0.7931,0.3622
...,...,...,...,...,...,...,...,...,...,...
163,gpt-5.4-mini,ZS,niveau2.8,definition,0,0,183,0.0000,0.0000,0.0000
164,gpt-5.4-mini,ZS,niveau2.8,exemple,0,0,241,0.0000,0.0000,0.0000
165,gpt-5.4-mini,ZS,niveau2.8,reference,0,0,29,0.0000,0.0000,0.0000
166,gpt-5.4-mini,ZS,niveau2.8,renvoi,0,0,56,0.0000,0.0000,0.0000


In [ ]:
# --- Vue pivotée du premier dataframe : micro F-mesure par tag, pour chaque (modele, type de prompt et niveau) ---
pivot_f1 = df_micro.pivot_table(
    index=["modele", "type_prompt", "niveau"],
    columns="tag",
    values="f1_micro"
).round(4)

pivot_f1

tag                                                  auteur  bibliographie  \
modele                        type_prompt niveau                             
gemini-3.1-flash-lite-preview FS          niveau2.6  0.8425         0.5522   
                                          niveau2.8  0.8185         0.3094   
                              FS-R        niveau2.6  0.5823         0.7961   
                                          niveau2.8  0.6393         0.6494   
                              ZS          niveau2.6  0.7259         0.4756   
                                          niveau2.8  0.5897         0.2411   
gemma-3-27b-it                FS          niveau2.6  0.2632         0.1538   
                                          niveau2.8  0.0357         0.0000   
                              FS-R        niveau2.6  0.4977         0.3611   
                                          niveau2.8  0.5236         0.2647   
                              ZS          niveau2.6  0.0000         0.0000   
                                          niveau2.8  0.0000         0.0000   
gpt-5-mini                    FS          niveau2.6  0.1461         0.0168   
                                          niveau2.8  0.4865         0.3380   
                              FS-R        niveau2.6  0.1143         0.0813   
                                          niveau2.8  0.0240         0.0333   
                              ZS          niveau2.6  0.0000         0.0000   
                                          niveau2.8  0.0000         0.0000   
gpt-5.4-mini                  FS          niveau2.6  0.3171         0.2330   
                                          niveau2.8  0.1250         0.1270   
                              FS-R        niveau2.6  0.2194         0.2385   
                                          niveau2.8  0.2162         0.0000   
                              ZS          niveau2.6  0.0000         0.0000   
                                          niveau2.8  0.0000         0.0000   

tag                                                  definition  exemple  \
modele                        type_prompt niveau                           
gemini-3.1-flash-lite-preview FS          niveau2.6      0.7054   0.7682   
                                          niveau2.8      0.9027   0.7422   
                              FS-R        niveau2.6      0.8098   0.7377   
                                          niveau2.8      0.7843   0.7265   
                              ZS          niveau2.6      0.8847   0.7330   
                                          niveau2.8      0.8777   0.6737   
gemma-3-27b-it                FS          niveau2.6      0.6084   0.2103   
                                          niveau2.8      0.4292   0.2344   
                              FS-R        niveau2.6      0.3183   0.3890   
                                          niveau2.8      0.5801   0.2562   
                              ZS          niveau2.6      0.0000   0.0000   
                                          niveau2.8      0.0000   0.0000   
gpt-5-mini                    FS          niveau2.6      0.6397   0.6059   
                                          niveau2.8      0.7774   0.6188   
                              FS-R        niveau2.6      0.5347   0.3827   
                                          niveau2.8      0.5816   0.1533   
                              ZS          niveau2.6      0.3421   0.0000   
                                          niveau2.8      0.0000   0.0000   
gpt-5.4-mini                  FS          niveau2.6      0.8656   0.4101   
                                          niveau2.8      0.9086   0.3578   
                              FS-R        niveau2.6      0.6148   0.2045   
                                          niveau2.8      0.7553   0.5232   
                              ZS          niveau2.6      0.0000   0.0000   
                                          niveau2.8      0.0000   0.000